In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [52]:
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import keras
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import LeakyReLU
from keras.utils import to_categorical

In [101]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import load_model

(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [121]:
x_train = x_train.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

In [128]:
y_test.shape

(10000, 10)

In [259]:
class kerasmodel:
    def __init__(self,xtrain, xtest, valtrain, valtest):
        self.x_train = xtrain
        self.x_test = xtest
        self.val_train = valtrain
        self.val_test = valtest
        self.train_result_loss = None
        self.train_result_acc = None
        self.test_result_loss = None
        self.test_result_acc = None
        self.modelk = None

    def createmodel(self):
        modelkeras = keras.Sequential([
            keras.layers.Dense(128, input_shape = (784,)),
            keras.layers.Dense(64, activation = 'relu'),
            keras.layers.Dense(32, activation = 'relu'),
            keras.layers.Dense(10, activation = 'softmax')
        ])
        modelkeras.compile(
        optimizer = keras.optimizers.Adam(learning_rate = 0.01),
        loss='categorical_crossentropy',
        metrics=['accuracy']
        )
        self.modelk = modelkeras
        return modelkeras


    def modelfit(self,batchsize = 64, epochs = 20):
        if self.modelk is None:
            self.createmodel()
        self.modelk.fit(self.x_train, self.val_train, batch_size = batchsize, epochs = epochs,verbose = 0)

    def train(self):
        if self.modelk is None:
            self.createmodel()
        train_loss, train_accuracy = self.modelk.evaluate(self.x_train, self.val_train)
        self.train_result_loss = train_loss
        self.train_result_acc = train_accuracy
        return train_loss, train_accuracy

    def test(self):
        if self.modelk is None:
            self.createmodel()
        test_loss, test_accuracy = self.modelk.evaluate(self.x_test, self.val_test)
        self.test_result_loss = test_loss
        self.test_result_acc = test_accuracy
        return test_loss, test_accuracy

    def save_model(self, name):
        self.modelk.save(name)

In [260]:
modelkeras = kerasmodel(x_train, x_test, y_train, y_test)

In [261]:
modelkeras.createmodel()

<Sequential name=sequential_9, built=True>

In [125]:
modelkeras.modelfit(batchsize = 64, epochs = 20)

In [126]:
modelkeras.train()

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9808 - loss: 0.0610


(0.06100618466734886, 0.9808499813079834)

In [127]:
modelkeras.test()

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9665 - loss: 0.1328    


(0.1327570378780365, 0.9664999842643738)

In [182]:
(X_train, val_train), (X_test, val_test) = mnist.load_data()

In [183]:
def data_for_torch(X_train, val_train, X_test, val_test):
    X_train = X_train.reshape(-1, 28*28)
    X_test = X_test.reshape(-1, 28*28)
    X_train = X_train.astype('float32') / 255
    X_test = X_test.astype('float32') / 255
    X_train = torch.tensor(X_train)
    X_test = torch.tensor(X_test)
    val_train = torch.tensor(val_train, dtype = torch.long)
    val_test = torch.tensor(val_test, dtype = torch.long)
    return X_train, val_train, X_test, val_test


def loader_for_torch(X_train, val_train, X_test, val_test):
    train = DataLoader(
        TensorDataset(X_train, val_train),
        batch_size = 64,
        shuffle = True
    )
    test = DataLoader(
        TensorDataset(X_test, val_test),
        batch_size = 64,
        shuffle = False
    )
    return train, test

X_train, val_train, X_test, val_test = data_for_torch(X_train, val_train, X_test, val_test)
trainloader, testloader = loader_for_torch(X_train, val_train, X_test, val_test)

In [184]:
print(X_train.shape)
print(X_test.shape)
print(val_train.shape)
print(val_test.shape)

torch.Size([60000, 784])
torch.Size([10000, 784])
torch.Size([60000])
torch.Size([10000])


In [236]:
class torchmodel(nn.Module):
    def __init__(self,input_size = X_train.shape[1]):
        super(torchmodel,self).__init__()
        self.sloi1 = nn.Linear(input_size, 128)
        self.sloi2 = nn.Linear(128, 64)
        self.activator1 = nn.ReLU()
        self.sloi3 = nn.Linear(64, 32)
        self.activator2 = nn.ReLU()
        self.sloi4 = nn.Linear(32, 10)


    def forward(self, x):
        out = self.sloi1(x)
        out = self.sloi2(out)
        out = self.activator1(out)
        out = self.sloi3(out)
        out = self.activator2(out)
        out = self.sloi4(out)
        return out

modeltorch = torchmodel()
lert = 0.001
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modeltorch.parameters(), lr = lert)
epoch = 20

In [237]:
def trainmodel(loader, model, loss_fn, optimizer):
    model.train()
    step = len(loader)
    n_epoch = 20

    for epoch in range(n_epoch):
        train_samples_count = 0 # общее количество картинок, которые мы прогнали через модельку
        true_train_samples_count = 0 # количество верно предсказанных картинок
        running_loss = 0
        for i, (batch, label) in enumerate(loader):
            pred = model(batch)

            loss = loss_fn(pred, label)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            running_loss += loss.item()
            pred = pred.argmax(dim=1, keepdim=False)
            true_classified = (pred == label).sum().item() # количество верно предсказанных картинок в текущем батче
            true_train_samples_count += true_classified
            train_samples_count += len(batch) # размер текущего батча

        train_accuracy = true_train_samples_count / train_samples_count
        print(f"[{epoch}] train loss: {running_loss}, accuracy: {round(train_accuracy, 4)}")

In [238]:
trainmodel(trainloader, modeltorch, loss_fn, optimizer )

[0] train loss: 374.27212914451957, accuracy: 0.8835
[1] train loss: 176.53822696954012, accuracy: 0.9435
[2] train loss: 135.26249743625522, accuracy: 0.9569
[3] train loss: 111.93271558452398, accuracy: 0.964
[4] train loss: 98.34870910085738, accuracy: 0.9683
[5] train loss: 87.87046129070222, accuracy: 0.9717
[6] train loss: 77.78497409680858, accuracy: 0.974
[7] train loss: 70.52145083225332, accuracy: 0.9762
[8] train loss: 64.62312527978793, accuracy: 0.9777
[9] train loss: 58.796186334919184, accuracy: 0.9792
[10] train loss: 55.32170936872717, accuracy: 0.9808
[11] train loss: 49.13849043659866, accuracy: 0.9827
[12] train loss: 45.270021960139275, accuracy: 0.9846
[13] train loss: 43.299163975403644, accuracy: 0.9845
[14] train loss: 40.3244789156015, accuracy: 0.9861
[15] train loss: 37.286155601381324, accuracy: 0.9865
[16] train loss: 34.21517340090941, accuracy: 0.9877
[17] train loss: 32.36431063490454, accuracy: 0.9886
[18] train loss: 29.330896923143882, accuracy: 0.98

In [250]:
def testmodel(loader, model, loss_fn, epoch):
    model.eval()
    secloss = 0
    corr = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            pred = model(images)
            loss = loss_fn(pred, labels)
            secloss += loss.item() * len(images) 

            pred = torch.argmax(pred, dim=1)
            corr += (pred == labels).sum().item()
            total_samples += len(labels)
            
    floss = secloss / total_samples
    accuracy = corr / total_samples
    print(f"[Epoch {epoch}] test loss: {floss:.4f}, accuracy: {accuracy:.4f}")

In [251]:
testmodel(testloader, modeltorch, loss_fn, epoch)

[Epoch 20] test loss: 0.1231, accuracy: 0.9736


In [262]:
modelkeras.save_model('keras_model.h5')

In [264]:
torch.save(modeltorch, 'torchmodel.pth')